In [1]:
from datasets import load_dataset

arc_dataset = load_dataset("allenai/ai2_arc", "ARC-Challenge")

train_set = arc_dataset['train']
test_set = arc_dataset['test']
validation_set = arc_dataset['validation']

import pandas as pd

def dataset_to_dataframe(set):
    
    rows = []
    
    skipped = 0
    
    for row in set:
        
        choices = row["choices"]["text"]
        
        if len(choices) < 4:
            skipped += 1
            continue
        
        rows.append({
            "question": row["question"],
            "option_A": choices[0],
            "option_B": choices[1],
            "option_C": choices[2],
            "option_D": choices[3],
            "answer_key": row["answerKey"]
        })
        
    return pd.DataFrame(rows), skipped

train_set, skipped_train = dataset_to_dataframe(train_set)
test_set, skipped_test = dataset_to_dataframe(test_set)
validation_set, skipped_val = dataset_to_dataframe(validation_set)

print(f"Total skipped: {skipped_train + skipped_test + skipped_val}")

full_set = pd.concat([train_set, test_set, validation_set], ignore_index=True)

full_set.head(5)

Total skipped: 8


,question,option_A,option_B,option_C,option_D,answer_key
0,George wants to warm his hands quickly by rubb...,dry palms,wet palms,palms covered with oil,palms covered with lotion,A
1,Which of the following statements best explain...,The refrigerator door is smooth.,The refrigerator door contains iron.,The refrigerator door is a good conductor.,The refrigerator door has electric wires in it.,B
2,A fold observed in layers of sedimentary rock ...,cooling of flowing magma.,converging of crustal plates.,deposition of river sediments.,solution of carbonate minerals.,B
3,Which of these do scientists offer as the most...,worldwide disease,global mountain building,rise of mammals that preyed upon plants and an...,impact of an asteroid created dust that blocke...,D
4,A boat is acted on by a river current flowing ...,west,east,north,south,B


In [13]:
import laya
agent = laya.load("convaiinnovations/laya")

def build_laya_question(row):
    return {
        "answer": {
            "type": "choice",
            "instructions": "Select the correct answer.",
            "criteria": {
                "A": row["option_A"],
                "B": row["option_B"],
                "C": row["option_C"],
                "D": row["option_D"],
            },
        }
    }
    
sample_row = full_set.iloc[0]
sample_result = agent.system_one(state="", questions=build_laya_question(sample_row))

print(sample_result, type(sample_result))

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

c:\Users\Jash\AppData\Local\Programs\Python\Python311\Lib\site-packages\laya\agent.py:928: RuntimeWarning: laya: this checkpoint ships invalid temperatures or values outside [0.5, 5]; using choice:11+=0.10058280825614929 -> 0.5. Treat confidence from the affected entries as uncalibrated.
  return Agent(model_id_or_path, device=device, token=token, subfolder=subfolder, fast=fast,


{'model': 'laya-rl-agent', 'answers': {'answer': {'type': 'choice', 'choice': 'C', 'probabilities': {'A': 0.2648, 'B': 0.1941, 'C': 0.3479, 'D': 0.1932}, 'confidence': 0.0226, 'answer_confidence': 0.3479, 'action': {'act_probability': 1.0}}}, 'usage': {'input_tokens': 37, 'output_tokens': 0}} <class 'dict'>


In [14]:
print(sample_result['model'])
print(sample_result['answers'])
print(sample_result['usage'])
print(sample_result['answers'].keys())

answer_dict = sample_result['answers']['answer']

print(answer_dict, answer_dict.keys())

print("=======")

for key in list(answer_dict.keys()):
    print(f"{key}: {answer_dict[key]}")
    
print(f"Answer: {answer_dict['choice']}")
print(f"Prob: {answer_dict['answer_confidence']}")

laya-rl-agent
{'answer': {'type': 'choice', 'choice': 'C', 'probabilities': {'A': 0.2648, 'B': 0.1941, 'C': 0.3479, 'D': 0.1932}, 'confidence': 0.0226, 'answer_confidence': 0.3479, 'action': {'act_probability': 1.0}}}
{'input_tokens': 37, 'output_tokens': 0}
dict_keys(['answer'])
{'type': 'choice', 'choice': 'C', 'probabilities': {'A': 0.2648, 'B': 0.1941, 'C': 0.3479, 'D': 0.1932}, 'confidence': 0.0226, 'answer_confidence': 0.3479, 'action': {'act_probability': 1.0}} dict_keys(['type', 'choice', 'probabilities', 'confidence', 'answer_confidence', 'action'])
type: choice
choice: C
probabilities: {'A': 0.2648, 'B': 0.1941, 'C': 0.3479, 'D': 0.1932}
confidence: 0.0226
answer_confidence: 0.3479
action: {'act_probability': 1.0}
Answer: C
Prob: 0.3479


In [15]:
def laya_predict(row):
    question = build_laya_question(row)
    result = agent.system_one(state=row["question"], questions=question)
    answer_dict = result["answers"]["answer"]
    choice = answer_dict["choice"]
    prob = answer_dict["answer_confidence"]
    
    return (choice, prob) 

In [16]:
from tqdm import tqdm

def laya_arc_inference(n_rows=None):
    
    set = full_set.head(n_rows) if n_rows else full_set
    
    correct = 0
    prob_total = 0
    
    for _, row in tqdm(set.iterrows(), total=len(set)):
        
        choice, prob = laya_predict(row)
        
        if choice == row["answer_key"]:
            correct += 1
            prob_total += prob
    
    print(f"Accuracy: {correct /  len(set):.3f}")
    print(f"Mean correct confidence: {prob_total / correct:.3f}")
    
laya_arc_inference(n_rows=None)

100%|██████████| 2582/2582 [01:22<00:00, 31.45it/s]

Accuracy: 0.314
Mean correct confidence: 0.489
